In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(dotenv_path=".env", override=True)

mykeys = os.getenv("KEY")


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=mykeys,
    temperature=0.3
)

quiz_template = """
Text:{text}

You are an expert MCQ maker.

Create {number} multiple choice questions for {subject} students in {tone} tone.

### RESPONSE_JSON
{response_json}
"""

quiz_prompt = PromptTemplate(
    input_variables=["text","number","subject","tone","response_json"],
    template=quiz_template
)

review_template = """
You are an expert English writer.

Evaluate the complexity of the following quiz for {subject} students.

Quiz:
{quiz}
"""

review_prompt = PromptTemplate(
    input_variables=["subject","quiz"],
    template=review_template
)

parser = StrOutputParser()

quiz_chain = quiz_prompt | llm | parser
review_chain = review_prompt | llm | parser

quiz = quiz_chain.invoke({
    "text":"AI transferme",
    "number":5,
    "subject":"computer science",
    "tone":"simple",
    "response_json":"{}"
})

review = review_chain.invoke({
    "subject":"Computer Science",
    "quiz":quiz
})

print("QUIZ:\n",quiz)
print("\nREVIEW:\n",review)

QUIZ:
 ```json
{
  "questions": [
    {
      "question": "What is the main idea behind 'AI transferme' in the context of machine learning?",
      "options": {
        "A": "Training an AI model from scratch for every new task.",
        "B": "Reusing a pre-trained AI model for a new, related task.",
        "C": "Transferring data between different AI models.",
        "D": "Manually programming AI rules for specific tasks."
      },
      "answer": "B"
    },
    {
      "question": "Why might a computer scientist choose to 'transfer' an AI model's knowledge instead of building a new one?",
      "options": {
        "A": "It always guarantees perfect accuracy.",
        "B": "It helps save time and computational resources, especially with limited data.",
        "C": "It makes the AI model smaller in size.",
        "D": "It's the only way to train deep learning models."
      },
      "answer": "B"
    },
    {
      "question": "When using 'AI transferme' for image recognition, w

In [ ]:
import json

quiz_table_data = []
raw_data = json.loads(quiz.replace("```json","").replace("```","").strip())
for i in range(len(raw_data['questions'])):
    item = raw_data["questions"][i]

    options_string = " | ".join(item["options"])

    correct_letters = item["answer"].split(')')[0]

    quiz_dict = {
        "MCQ": item["question"],
        "Options": options_string,
        "Correct": correct_letters
    }
    quiz_table_data.append(quiz_dict)
print("[")
for i in range(len(quiz_table_data)):
    print(f"{quiz_table_data[i]},")
    
print("]")

[
{'MCQ': 'According to the text, what is AI doing to technology?', 'Options': 'a) Ignoring it | b) Slowing it down | c) Changing it significantly | d) Keeping it the same', 'Correct': 'c'},
{'MCQ': "When the text says 'AI is transforming technology,' what does 'transforming' primarily imply?", 'Options': 'a) Making minor adjustments | b) Causing fundamental and widespread changes | c) Only affecting a few specific areas | d) Reversing previous progress', 'Correct': 'b'},
{'MCQ': "In the statement 'AI is transforming technology,' which part is the *agent* or *cause* of the transformation?", 'Options': 'a) Technology | b) AI | c) Transforming | d) The statement itself', 'Correct': 'b'},
{'MCQ': 'AI, or Artificial Intelligence, is a major field within which broader academic discipline?', 'Options': 'a) Biology | b) Computer Science | c) Literature | d) Economics', 'Correct': 'b'},
{'MCQ': "If technology is being 'transformed' by AI, what would be a likely characteristic of this transform

In [7]:
quiz_table_data

[{'MCQ': 'According to the text, what is AI doing to technology?',
  'Options': 'a) Ignoring it | b) Slowing it down | c) Changing it significantly | d) Keeping it the same',
  'Correct': 'c'},
 {'MCQ': "When the text says 'AI is transforming technology,' what does 'transforming' primarily imply?",
  'Options': 'a) Making minor adjustments | b) Causing fundamental and widespread changes | c) Only affecting a few specific areas | d) Reversing previous progress',
  'Correct': 'b'},
 {'MCQ': "In the statement 'AI is transforming technology,' which part is the *agent* or *cause* of the transformation?",
  'Options': 'a) Technology | b) AI | c) Transforming | d) The statement itself',
  'Correct': 'b'},
 {'MCQ': 'AI, or Artificial Intelligence, is a major field within which broader academic discipline?',
  'Options': 'a) Biology | b) Computer Science | c) Literature | d) Economics',
  'Correct': 'b'},
 {'MCQ': "If technology is being 'transformed' by AI, what would be a likely characteristi

In [12]:
import pandas as pd

quiz = pd.DataFrame(quiz_table_data)



In [13]:
import csv
quiz.to_csv("machine learning.csv", index=False)